# Entrenamiento LoRA — Text-to-SQL (Colab, GPU T4)

Entrena las dos configuraciones, que difieren en un solo factor (**módulos con LoRA**):

| Config | Módulos con LoRA | Resto |
|---|---|---|
| A | `q_proj, k_proj, v_proj, o_proj` (atención) | r=16, alpha=32, dropout=0.05, lr=2e-4, 1 época, batch efectivo 16 |
| B | atención + `gate_proj, up_proj, down_proj` (MLP) | igual |

**Hipótesis (escrita antes de entrenar):** B logrará menor loss en el dev interno y mayor EX que A. Las capas MLP guardan buena parte del conocimiento de patrones (sintaxis MySQL, funciones de fecha, ventanas), y adaptarlas debería ayudar más que adaptar solo la atención. A cambio, B tendrá ~2.5× más parámetros entrenables y tardará más por paso.

Antes de empezar: *Entorno de ejecución → Cambiar tipo → T4 GPU*. Guarda un token de GitHub en *Secrets* con el nombre `GH_TOKEN`.

**Si Colab se desconecta:** los checkpoints se guardan cada 50 pasos en Google Drive (`MyDrive/t2s_checkpoints/`). Vuelve a correr todas las celdas desde arriba y el entrenamiento continúa donde se quedó. Cada configuración se sube al repo apenas termina.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINTS = '/content/drive/MyDrive/t2s_checkpoints'

In [ ]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git clone -q https://{token}@github.com/Qu3zada22/proyecto3-deep.git
%cd proyecto3-deep
!git pull -q
!pip install -q -r requirements.txt
!git config user.name 'Qu3zada22' && git config user.email '139565119+Qu3zada22@users.noreply.github.com'

## Configuración A

In [ ]:
# si ya terminó en una sesión anterior (y se subió al repo), no se vuelve a entrenar
![ -f resultados/entrenamiento/A/resumen.json ] && echo 'Config A ya entrenada' || python scripts/entrenamiento/entrenar.py --config A --checkpoints {CHECKPOINTS}

In [ ]:
!git add resultados/entrenamiento/A && git commit -q -m 'Resultados de entrenamiento LoRA A (Colab T4)' && git push -q

## Configuración B

In [ ]:
# si ya terminó en una sesión anterior (y se subió al repo), no se vuelve a entrenar
![ -f resultados/entrenamiento/B/resumen.json ] && echo 'Config B ya entrenada' || python scripts/entrenamiento/entrenar.py --config B --checkpoints {CHECKPOINTS}

In [ ]:
!git add resultados/entrenamiento/B && git commit -q -m 'Resultados de entrenamiento LoRA B (Colab T4)' && git push -q

## Curvas de loss

In [ ]:
from IPython.display import Image, display
import json
for c in 'AB':
    print(c, json.load(open(f'resultados/entrenamiento/{c}/resumen.json')))
    display(Image(f'resultados/entrenamiento/{c}/curva_loss.png'))